# Olist Brazilian E-Commerce — SQL Analysis
**Dataset :** Kaggle Olist Brazilian E-Commerce (99 441 orders, 6 tables)  
**Objectif :** Identifier les catégories, vendeurs et régions les plus performants  
**Stack :** Python, SQLite, Pandas

All amounts in BRL (Brazilian Real)

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("olist.db")

def run_query(sql):
    return pd.read_sql(sql, conn)

print("Connexion établie ✓")
print("Tables disponibles :")
print(run_query("SELECT name FROM sqlite_master WHERE type='table'"))

Connexion établie ✓
Tables disponibles :
                   name
0             customers
1                orders
2           order_items
3              products
4               sellers
5  category_translation


## Question 1 : Top 10 des catégories par revenu total
On va d'abord rechercher les catégories de produit qui rapportent le plus de revenu afin de se donner un ordre d'idée pour les analyses suivantes. Étant donné que les noms de la table produit sont en portugais on va lier la table "products" avec la table "category_translation", puis rechercher les catégories de produits rapportant le plus avec un tri. On remarque que la catégorie dominante est Health & Beauty avec 1,25MBRL de revenu, suivie par Watches&Gifts avec 1,20MBRl puis Bed,Bath&Table avec 1MBRL, cela pourrait laisser penser que même si la catégorie dominante est constituée de produits du quotidien achetés régulièrement (volume), elle est suivie de près par des catégories à forte valeur ou mid-range. 
    **Resultats : 1er =  health_beauty     1 258 681BRL, 2ème = watches_gifts     1 205 005BRL, 3ème =   bed_bath_table     1 036 988BRL**

q1 = run_query("""
    SELECT ct.product_category_name_english AS category,
           SUM(oi.price) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print(q1)

## Question 2 : Top 10 des vendeurs les plus performants
On recherche ensuite les meilleurs vendeurs pour pouvoir approfondir les recherches suivantes. Pour ce faire, on recherche simplement quels vendeurs ont generé le plus de revenus . On remarque que le vendeur numéro 1 a dominé par le volume avec 1156 commandes pour un revenu de 229 472BRL, comme pour le numéro 3, tandis que le numéro 2 avec 410 commandes et 222 776BRL a plutôt dominé par la valeur des produits vendus. Ces tendances distinguent deux types de vendeurs : ceux qui se concentrent sur le volume et ceux qui preferent les articles à plus forte valeur.
    **Resultats : 1er =  1156 commandes pour 229 472,63BRL; 2ème = 410 commandes pour 222 776,05BRL; 3ème = 1987 commandes pour 200 472,92BRL**

In [2]:
q2 = run_query("""
    SELECT seller_id,
           COUNT(order_id) AS nb_commandes,
           SUM(price) AS total_ventes
    FROM order_items
    GROUP BY seller_id
    ORDER BY total_ventes DESC
    LIMIT 10
""")

print(q2)

                          seller_id  nb_commandes  total_ventes
0  4869f7a5dfa277a7dca6462dcf3b52b2          1156     229472.63
1  53243585a1d6dc2643021fd1853d8905           410     222776.05
2  4a3ca9315b744ce9f8e9374361493884          1987     200472.92
3  fa1c13f2614d7b5c4749cbc52fecda94           586     194042.03
4  7c67e1448b00f6e969d365cea6b010ab          1364     187923.89
5  7e93a43ef30c4f03f38b393420bc753a           340     176431.87
6  da8622b14eb17ae2831f4ac5b9dab84a          1551     160236.57
7  7a67c85e85bb2ce8582c35f2203ad736          1171     141745.53
8  1025f0e2d44d7041d6cf58b6550e0bfa          1428     138968.55
9  955fee9216a65b617aa5c0531780ce60          1499     135171.70


## Question 3 : Panier moyen par vendeur
On cherche à determiner le panier moyen par vendeur. Pour ce faire, on cherche la moyenne des dépenses de chaque vendeur, puis on les classe. On remarque que le vendeur avec le panier moyen le plus élévé dominé par la valeur du panier puisqu'il n'y a qu'un seul article d'une valeur de 6 729BRL comme pour les 2ème et 3ème paniers. Cependant, plus on descend plus on constate une hausse du nombre d'articles dans les paniers, ce qui rejoins l'idée que certains vendeurs se distinguent plus par le volume vendu. 
    **Resultats : 1er =  1 commande pour 6729BRL; 2ème = 1 commande pour  6499BRL; 3ème = 1 commande pour 3549BRL**

In [3]:
q3 = run_query("""
    SELECT seller_id, 
        COUNT(order_id) AS nb_commandes , 
        AVG(price) AS panier_moyen
    FROM order_items
    GROUP BY seller_id
    ORDER BY panier_moyen DESC 
    LIMIT 10 
""")

print(q3)

                          seller_id  nb_commandes  panier_moyen
0  80ceebb4ee9b31afb6c6a916a574a1e2             1   6729.000000
1  ee27a8f15b1dded4d213a468ba4eb391             1   6499.000000
2  585175ec331ea177fa47199e39a6170a             1   3549.000000
3  abe021b01ba992245271b9aa422032df             2   3360.000000
4  a00824eb9093d40e589b940ec45c4eb0             3   3133.323333
5  e2a1ac9bf33e5549a2a4f834e70df2f8             5   2999.890000
6  e908c0f3646e8b60375734a350d95d71             1   2951.000000
7  d63c73efd41eb002280e7ec831424edb             2   2799.000000
8  1444c08e64d55fb3c25f0f09c07ffcf2             1   2749.000000
9  039e6ad9dae79614493083e241147386             5   2585.000000


## Question 4 : Dans quels États les clients dépensent-ils le plus ?
L'objectif est ici de distinguer les zones géographiques qui supportent le plus de commandent de la part des clients afin de faire ressortir des points chauds. Pour cela on va devoir chercher le chiffre d'affaire et le panier moyen dans chacun des États, puis lier les tables "orders" et "order_items" à la table "customers". On remarque qu'une zone, São Paulo, se démarque fortement des autres, et même du second avec  47 449 commandes et un chiffre d'affaire maximum de 5 202 955,05BRL. Tandis que le second, Rio de Janeiro, ne comptabilise que 14 579 commandes pour un CAmax de 1 824 092,67BRL. Cela s'explique sûrement par le fait que São Paulo est la capitale économique du pays, et concentre donc potentiellement plus d'acheteurs avec un plus large portefeuille. 
    **Resultats : 1er =  SP avec 47 449 commandes et un CA de 5 202 955,05BRL; 2ème = RJ 14 579 commandes et un CA de 1 824 092,67BRL; 3ème = MG avec 13 129 commandes et un CA de 1 585 308,03BRL**

    
## Question 4 (bonus 1): SP commande plus en quantité ou en prix ?
Dans le cas de São Paulo, on cherche ici à savoir si le CA repose plus sur un facteur de valeur ou de volume par rapport aux autres États. En l'occurence on observe que alors que le panier moyen de SP est de 109,653BRL, celui du deuxième État, Rio de Janeiro, est de 125,117BRL. On peut donc constater que le CA de SP repose plus sur un facteur volume que valeur. 
    **Resultats : 1er = SP avec un panier moyen de 109,65BRL; 2ème = RJ avec un panier moyen de 125,118BRL**
    
## Question 4 (bonus 2): Quelles sont les top catégories à SP ?
Afin de confirmer notre théorie sur le fait que le CA de São Paulo repose principalement sur un facteur volume on va distinguer les top catégories de la zone. À la maniere des questions 1 et 4 on lie les tables "order_items" et "orders", qui contiennent les id des commandes, à la table "customers" qui contient les id des clients. Puis lier les tables "order_items" et "products", qui contiennent les id des produits, à la table "category_translation" qui contient la traduction anglaise des catégories de produits. À partir de ces données on constate que les deux catégories dominantes sont Bed&Bath&Table avec 5 235 commandes, un CA de 478 284BRL et un panier moyen de 91,36BRL, et Health&Beauty avec 4 204 commandes, un CA de 462 305BRL et un panier moyen de 109,96 BRL. Deux catégories qui se reposent plus sur le volume. Cependant, la troisième catégorie, Watches&Gifts, comporte 2 281 commandes, un CA de 435 009BRL et un panier moyen de 190BRL, nous montre l'importance du facteur valeur avec un panier moyen nettement plus élévé que les deux premières.  
    **Resultats : 1er =  bed_bath_table, 5 235 commandes, CA 478 284,52 BRL, PM 91,36BRL; 3ème = watches_gifts, 2281 commandes, CA 435 009,92 BRL, PM 190,71BRL**

In [4]:
q4 = run_query("""
    SELECT customer_state,
        COUNT(oi.order_id) AS nb_commandes, 
        SUM(price) AS max_CA,
        AVG(oi.price) AS panier_moyen
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY customer_state
    ORDER BY max_CA DESC
    LIMIT 10 
""")

print(q4)


q4_bonus2 = run_query("""
    SELECT customer_state, ct.product_category_name_english AS category,
        COUNT(oi.order_id) AS nb_commandes, 
        SUM(price) AS max_CA,
        AVG(oi.price) AS panier_moyen 
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    GROUP BY customer_state, category
    ORDER BY max_CA DESC
    LIMIT 10 
""")

print(q4_bonus2)

  customer_state  nb_commandes      max_CA  panier_moyen
0             SP         47449  5202955.05    109.653629
1             RJ         14579  1824092.67    125.117818
2             MG         13129  1585308.03    120.748574
3             RS          6235   750304.02    120.337453
4             PR          5740   683083.76    119.004139
5             SC          4176   520553.34    124.653578
6             BA          3799   511349.99    134.601208
7             DF          2406   302603.94    125.770549
8             GO          2333   294591.95    126.271732
9             ES          2256   275037.31    121.913701


  customer_state               category  nb_commandes     max_CA  panier_moyen
0             SP         bed_bath_table          5235  478284.52     91.362850
1             SP          health_beauty          4204  462305.22    109.967940
2             SP          watches_gifts          2281  435009.92    190.710180
3             SP         sports_leisure          3667  386357.01    105.360515
4             SP  computers_accessories          3170  350747.88    110.646019
5             SP        furniture_decor          3531  286708.02     81.197400
6             SP             housewares          3265  275378.63     84.342613
7             SP                   auto          1747  214277.27    122.654419
8             SP             cool_stuff          1364  213186.89    156.295374
9             SP                   toys          1712  185561.83    108.388919


## Question 5 : Quelle catégorie domine chez les top vendeurs ?
De retour sur les vendeurs, l'objectif est de confirmer l'idée selon laquelle certains vendeurs se concentrent sur des catégories à forte valeur, tandis que certains préfèrent le volume, et d'ainsi distinguer lesquels sont les plus performants. Pour cela on va devoir créer une table temporaire que l'on nommera "top_seller" qui, comme son nom l'indique, classe les meilleurs vendeurs selon leurs CA. Puis, on la joindra à la table "order_items", qui contient aussi l'id des vendeurs, à la table "products" pour l'id des produits, et enfin la table "category_translation" qui contient la traduction du nom des catégories. On remarque que la catégorie Bed,Bath&Tables domine le podium, comme dans la question 1 où elle arrivait 3ème des catégories qui rapportent le plus avec un CA de 1 036 988BRL, avec les vendeurs numéro 1 et numéro 3 qui comptabilisent respectivement 1572 et 1277 ventes. Pour le vendeur numéro 2, la catégorie dominante est Furniture&Decor avec 1292 ventes, ce qui soutient l'idée précédente. On peut donc constater qu'étant donné que ces produits n'ont une valeur ni faible ni élevée, les vendeurs misent probablement sur un facteur volume avec des articles mid-range. 
    **Resultats : bed_bath_table avec 1572 ventes; 2ème = furniture_decor avec 1292 ventes ; 3ème =  bed_bath_table avec 1277 ventes**


In [5]:
q5 = run_query("""
    WITH top_sellers AS (
        SELECT seller_id, SUM(price) AS total_revenue
        FROM order_items
        GROUP BY seller_id
        ORDER BY total_revenue DESC
        LIMIT 10
    )
    SELECT ts.seller_id, ct.product_category_name_english AS category, 
            COUNT(*) AS nb_ventes    
    FROM top_sellers AS ts
    JOIN order_items oi ON ts.seller_id = oi.seller_id 
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    GROUP BY ts.seller_id, ct.product_category_name_english
    ORDER BY nb_ventes DESC
    LIMIT 10
""")

print(q5)


                          seller_id          category  nb_ventes
0  4a3ca9315b744ce9f8e9374361493884    bed_bath_table       1572
1  1025f0e2d44d7041d6cf58b6550e0bfa   furniture_decor       1292
2  da8622b14eb17ae2831f4ac5b9dab84a    bed_bath_table       1277
3  7c67e1448b00f6e969d365cea6b010ab  office_furniture       1233
4  7a67c85e85bb2ce8582c35f2203ad736        cool_stuff       1069
5  4869f7a5dfa277a7dca6462dcf3b52b2     watches_gifts       1002
6  fa1c13f2614d7b5c4749cbc52fecda94     watches_gifts        579
7  955fee9216a65b617aa5c0531780ce60   furniture_decor        535
8  7e93a43ef30c4f03f38b393420bc753a     watches_gifts        314
9  da8622b14eb17ae2831f4ac5b9dab84a     health_beauty        270


## Conclusion
Pour conclure cette analyse on peut distinguer plusieurs tendances. Dans un premier temps on remarque qu'il existe un veritable point chaud géographique, un État qui concentre une grande part des transactions, São Paulo. Avec 47 449 commandes et un chiffre d'affaire de 5 202 955,05BRL, qui n'est suivi que par les 14 579 commandes et les 1 824 092,67BRL de CA de Rio de Janeiro. Cela montre que son statut de capitale économique du pays fait de São Paulo le centre névralgique des échanges e-commerce. Pour ce qui est des catégories de produits on peut finalement distinguer trois catégories : les produits à faible valeur pour lesquels les volumes seront importants (Furniture&Decor), les produits à forte valeur et faible volume (Watches&Gifts) et une dernière catégorie qui réunit la valeur et le volume, des produits mid-range avec un fort volume (Bed,Bath&Table). Ainsi, comme vu dans la question 5, ce sont sur ce type de produits que vont se concentrer les meilleurs vendeurs car ils bénéficient à la fois de l'effet prix et de l'effet volume. Cette tendance se retrouve aussi dans les catégories de produits les plus vendus à SP avec une domination de ces mid-range comme Bed,Bath&Table, mais aussi de produits à forte récurrence d'achat comme Health&Beauty. Finalement, on peut critiquer cette analyse car elle ne détaille pas assez le prix individuel des articles dans chaque catégorie, et donc la valeur moyenne de la catégorie en question, ce qui rend difficile le fait d'établir une vraie échelle de valeur des différentes catégories. De plus, il est intéressant de noter que le dataset date de 2018 (pré-covid), ce qui nous permet de nous interroger sur l'évolution des ventes pendant et post-Covid, voire même de 2025 afin de dégager un trend. 